# BRAMASTRA K8: Two-T4 Cognition and Recursive Improvement Campaign

**Owner-launched.** Allocation: max **600 elapsed min / 1200 GPU-min** on 2x T4 (fits the 12-hour Kaggle GPU session cap with margin).

**Important:** kernel restarts do NOT reset the clock (SQLite ledger).
If E0 already ran, the full-run cell skips it.
Stop new training by minute 570. Hard stop before 600 (30-minute export reserve).

**Kaggle setup:** select accelerator `GPU T4 x2` and enable **Internet** in Session options. The notebook clones the pinned BRAMASTRA branch from GitHub when no source Dataset is attached, then builds the required K8 bundle when no prepared bundle Dataset is attached. Attached inputs remain supported for an offline/reproducible rerun.

For a new independent campaign, set `BRAMASTRA_RUN_ID` before running the setup cell. Leave it unchanged when resuming the same Kaggle session. Results are written under `/kaggle/working` and are retained when the notebook is saved as a Kaggle version.

**Run order:** 1 Bundle → 2 Validate → 3 Verify → 4 E0 Gate → 5 Full Campaign → 6 Summarize → 7 Export → 8 X-probe → 9 Package → 10 Results ZIP → 11 Safety sweep (setup cell runs first).


In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

WORKING = Path(os.environ.get('BRAMASTRA_WORKING', '/kaggle/working'))
INPUT = Path(os.environ.get('BRAMASTRA_INPUT', '/kaggle/input'))
WORKING.mkdir(parents=True, exist_ok=True)
GIT_URL = os.environ.get('BRAMASTRA_GIT_URL', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git')
GIT_REF = os.environ.get('BRAMASTRA_GIT_REF', 'BRAMASTRA')
AUTO_SOURCE_DIR = WORKING / 'bramastra-source'
MAX_WALL_MINUTES = int(os.environ.get('BRAMASTRA_MAX_WALL_MINUTES', '600'))

def _children(path):
    if not path.is_dir():
        return []
    try:
        return sorted(path.iterdir(), key=lambda item: item.name)
    except OSError:
        return []

def _source_candidates():
    configured = os.environ.get('BRAMASTRA_REPO')
    if configured:
        yield Path(configured)
    yield Path.cwd()
    yield WORKING / 'An-Ra-the-new-AGI'
    for parent in (INPUT, WORKING):
        for child in _children(parent):
            yield child
            for grandchild in _children(child):
                yield grandchild

def _is_source_tree(path):
    return (path / 'pyproject.toml').is_file() and (path / 'bramastra_lab').is_dir()

REPO = next((path.resolve() for path in _source_candidates() if _is_source_tree(path)), None)
if REPO is None:
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._/-]{0,127}', GIT_REF):
        raise RuntimeError('BRAMASTRA_GIT_REF contains unsafe characters.')
    if AUTO_SOURCE_DIR.exists():
        raise RuntimeError(
            f'{AUTO_SOURCE_DIR} already exists but is not a usable BRAMASTRA source tree. '
            'Use a new Kaggle session or set BRAMASTRA_REPO; the notebook will not delete it.')
    print(f'cloning BRAMASTRA {GIT_REF!r} from {GIT_URL}...')
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', GIT_REF, GIT_URL, str(AUTO_SOURCE_DIR)],
        capture_output=True, text=True)
    if clone.returncode != 0 or not _is_source_tree(AUTO_SOURCE_DIR):
        detail = (clone.stderr or clone.stdout).strip()[-1200:]
        raise RuntimeError(
            'Automatic BRAMASTRA clone failed. In Kaggle open Session options and enable Internet, '
            f'then start a fresh session. Git detail: {detail}')
    REPO = AUTO_SOURCE_DIR.resolve()
    revision = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
                              capture_output=True, text=True, check=True).stdout.strip()
    print('cloned source revision:', revision)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if importlib.util.find_spec('bramastra_lab') is None:
    raise RuntimeError(f'BRAMASTRA source exists at {REPO}, but Python cannot import it.')

REQUIRED_MODULES = ('torch', 'numpy', 'pytest')
missing_modules = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        'Kaggle runtime is missing required modules: ' + ', '.join(missing_modules) +
        '. Select a compatible Kaggle Python image or add the pinned dependency before starting this campaign.')

from bramastra_lab.research.campaigns.kaggle_env import (
    K8EnvironmentError,
    build_run_paths,
    package_artifacts,
    check_git_ref,
    discover_source,
    disk_contract,
    find_build_report,
    gpu_contract,
    stream_command,
)

try:
    PATHS = build_run_paths()
    RUN_ID = PATHS.run_id
    INSTANCE_ID = PATHS.instance_id
    RUN_ROOT = PATHS.run_root
    RUN_DIR = PATHS.run_dir
    BUILD_REPORT_DIR = PATHS.build_report_dir
    EXPORT_DIR = PATHS.export_dir
    WORKING = PATHS.working
    INPUT = PATHS.input_root
except K8EnvironmentError as exc:
    raise RuntimeError(f'campaign environment refused: {exc}') from exc


def run_k8(label, *arguments):
    try:
        result = stream_command(
            label, [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8', *arguments],
            cwd=REPO,
            log_dir=RUN_ROOT,
            extra_env={'PYTHONPATH': str(REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')},
        )
    except K8EnvironmentError as exc:
        raise RuntimeError(f'{label} could not start: {exc}') from exc
    if result.returncode != 0:
        try:
            _safety = subprocess.run(
                [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
                 'safety-snapshot', '--run-dir', str(RUN_ROOT / 'campaign' if 'RUN_DIR' not in dir() else RUN_DIR),
                 '--reason', f'failed-{label}'],
                cwd=REPO, capture_output=True, text=True, timeout=300)
            if _safety.stdout:
                print(_safety.stdout)
        except Exception as _safety_error:
            print(f'safety snapshot unavailable: {_safety_error}', file=sys.stderr)
        try:
            packed = package_artifacts(RUN_ROOT, WORKING, RUN_ID, INSTANCE_ID, f'failed-{label}')
            if packed is not None:
                print(f'failure artifacts packaged: {packed.archive}')
        except Exception as archive_error:
            print(f'could not package failure artifacts: {archive_error}', file=sys.stderr)
        try:
            disk = disk_contract(WORKING)
            print(f"disk free: {disk['free_gib']} GiB", file=sys.stderr)
        except K8EnvironmentError as disk_error:
            print(f'disk check failed: {disk_error}', file=sys.stderr)
        raise RuntimeError(f'{label} failed with exit code {result.returncode} (full log: {result.log_path})')
    return result

check_git_ref(GIT_REF)
try:
    _gpu = gpu_contract(required=2)
    print('GPU count:', _gpu['count'])
    for _i, _name in enumerate(_gpu['names']):
        print(f'  cuda:{_i}:', _name)
    print('torch:', _gpu['torch'])
except K8EnvironmentError as exc:
    raise RuntimeError(str(exc)) from exc
try:
    _disk = disk_contract(WORKING)
    print(f"disk free under {WORKING}: {_disk['free_gib']} GiB")
except K8EnvironmentError as exc:
    print(f'disk check unavailable: {exc}')
import torch
from bramastra_lab.research.runtime.provenance import source_identity
src = source_identity()
print('source:', json.dumps(src, indent=2))
print('source root:', REPO)
print('run ID:', RUN_ID)
print('Phases: E0(0-38) E1(38-190) E2(190-247) E3(247-323) E4(323-399) E5(399-570) E6(570-600)')
from bramastra_lab.research.config import tokenizer_identity
from bramastra_lab.research.campaigns.phases.ops import k8_campaign_config
cfg = k8_campaign_config()
print('config:', cfg.identity())
print('tokenizer:', tokenizer_identity())
print('model: vocab260/L8/W256/H4/FFN704/ctx512 params6493952')
print('allocation: single 600min campaign / 1200 provisioned GPU-min; E0 and full share it')


## 1. Acquire the K8 Data Bundle

An attached valid bundle takes precedence. When none is attached, this cell deterministically generates the full required bundle from the cloned source before any verification or GPU allocation begins.

In [ ]:
from bramastra_lab.research.campaigns.kaggle_env import discover_bundle, is_k8_bundle
BUNDLE_PATH, _bundle_checked = discover_bundle()
if BUNDLE_PATH is None:
    print('no attached bundle matched. INPUT tree seen:')
    for _child in sorted(INPUT.iterdir(), key=lambda item: item.name) if INPUT.is_dir() else []:
        print('  input/', _child.name)
        if _child.is_dir():
            for _grand in sorted(_child.iterdir(), key=lambda item: item.name)[:20]:
                print('    ', _grand.name)
    generated_bundle = RUN_ROOT / f'generated-k8-data-{INSTANCE_ID}'
    print('no attached K8 bundle found; generating the full deterministic bundle...')
    run_k8(
        'prepare K8 data', 'prepare', '--out', str(generated_bundle),
        '--training-mechanisms', '4096', '--controller-mechanisms', '256',
        '--development-mechanisms', '256', '--confirmation-mechanisms', '128',
        '--tool-mechanisms', '256', '--tool-heldout', '64',
        '--meta-train', '24', '--meta-validate', '6', '--meta-confirm', '6')
    if not is_k8_bundle(generated_bundle):
        raise RuntimeError('K8 preparation reported success but did not create a valid manifest.')
    BUNDLE_PATH = generated_bundle.resolve()
BUNDLE_DIR = str(BUNDLE_PATH)
print('using K8 bundle:', BUNDLE_DIR)


## 2. Validate Data

In [ ]:
run_k8('validate', 'validate', '--bundle', BUNDLE_DIR)


## 3. Build Verification (pre-allocation, zero optimizer commits)

Runs the registered local contract checks and real no-step production interfaces, then writes an evidence-backed build report. The campaign must not start unless the build verifies.


In [ ]:
_existing_report = BUILD_REPORT_DIR / 'build_verification.json'
if _existing_report.is_file():
    print('reusing existing build report:', _existing_report)
else:
    run_k8(
        'build verification', 'verify-build', '--data', BUNDLE_DIR,
        '--report-dir', str(BUILD_REPORT_DIR), '--no-updates',
        '--notebook', str(REPO / 'notebooks' / 'bramastra_k8.ipynb'))
print('build report:', BUILD_REPORT_DIR / 'build_verification.json')


## 4. E0 Gate

In [ ]:
run_k8(
    'E0 gate', 'run', '--mode', 'e0', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', str(MAX_WALL_MINUTES),
    '--build-report', str(find_build_report(RUN_ROOT, BUILD_REPORT_DIR)),
    '--precision', 'fp32')


## 5. Full Campaign (E1 through E6; E0 reused)
If E0 already ran, this cell skips it and uses remaining time.

In [ ]:
run_k8(
    'full campaign', 'run', '--mode', 'full', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', str(MAX_WALL_MINUTES),
    '--build-report', str(find_build_report(RUN_ROOT, BUILD_REPORT_DIR)),
    '--precision', 'fp32')


## 6. Summarize

In [ ]:
run_k8('summarize', 'summarize', '--run-dir', str(RUN_DIR))


## 7. Export

In [ ]:
run_k8('export', 'export', '--run-dir', str(RUN_DIR), '--out', str(EXPORT_DIR))
print('K8 export:', EXPORT_DIR)
print('Save a Kaggle version to retain this /kaggle/working output.')


## 8. X-factor probe pack (read-only architecture review)

Runs after export, before packaging: attention geometry, checkpoint lineage, allocation fidelity, phase accounting, tokenizer round trip. No allocation, zero optimizer updates, fresh report dir per run.

In [ ]:
import uuid as _uuid
_existing_xprobe = sorted(RUN_ROOT.glob('xprobe-*/xprobe_report.json'))
if _existing_xprobe:
    print('reusing xprobe report:', _existing_xprobe[-1])
    XPROBE_DIR = _existing_xprobe[-1].parent
else:
    XPROBE_DIR = RUN_ROOT / f'xprobe-{_uuid.uuid4().hex[:8]}'
    run_k8('xprobe', 'xprobe', '--run-dir', str(RUN_DIR), '--out', str(XPROBE_DIR), '--device', 'cuda:0')
_report_path = XPROBE_DIR / 'xprobe_report.json'
print('xprobe report:', _report_path)
_xr = json.loads(_report_path.read_text(encoding='utf-8'))
for _probe in _xr.get('probes', []):
    print(f"  {_probe['name']}: {_probe['status']}")
_p4 = next((x for x in _xr.get('probes', []) if x['name'] == 'P4-phase-accounting'), {})
if isinstance(_p4, dict) and 'total_device_minutes' in _p4:
    print(f"total device minutes: {_p4['total_device_minutes']}")
print('xprobe pass:', _xr.get('xprobe_pass'), 'failing:', _xr.get('failing'))


## 9. Package and Download All Artifacts

Creates one verified ZIP containing the complete generated run root: campaign ledger, build report, generated bundle when applicable, checkpoints, phase results, exports, and failure diagnostics.

In [ ]:
from bramastra_lab.research.campaigns.kaggle_env import package_artifacts
packed = package_artifacts(RUN_ROOT, WORKING, RUN_ID, INSTANCE_ID, 'completed')
if packed is None:
    raise RuntimeError('No K8 run artifacts exist yet. Run the campaign cells before packaging.')
ARCHIVE_PATH = packed.archive
print('Verified artifact archive:', ARCHIVE_PATH)
print('Archive receipt:', packed.receipt)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE_PATH)))
except Exception as exc:
    print(f'Use Kaggle Output to download {ARCHIVE_PATH.name}: {exc}')
print('Save a Kaggle version, then download the ZIP from the Output panel if the link is not shown.')


## 10. Results ZIP (all experiment results, ~10MB, auto-download)

Packs ledger export, phase outputs, build/xprobe reports, manifests and logs — no checkpoint payload bytes, so the ZIP stays small enough for the Output panel. The file link downloads on click; Save a Version also preserves it in Output.


In [ ]:
RESULTS_ZIP = RUN_ROOT / f'{RUN_ID}-results.zip'
if RESULTS_ZIP.is_file():
    print('reusing results pack:', RESULTS_ZIP)
else:
    _pack_args = ['package-results', '--run-dir', str(RUN_DIR), '--out', str(RESULTS_ZIP), '--run-id', RUN_ID]
    for _extra in (BUILD_REPORT_DIR, EXPORT_DIR):
        _pack_args += ['--extra', str(_extra)]
    for _xp in sorted(RUN_ROOT.glob('xprobe-*')):
        _pack_args += ['--extra', str(_xp)]
    run_k8('package results', *_pack_args)
_receipt = json.loads((RESULTS_ZIP.with_suffix('.json')).read_text(encoding='utf-8'))
print(f"results pack: {RESULTS_ZIP.name} {_receipt['bytes'] // 1024} KiB, {_receipt['files']} files")
print('sha256:', _receipt['sha256'][:16] + '...')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULTS_ZIP)))
except Exception as exc:
    print(f'Use Kaggle Output to download {RESULTS_ZIP.name}: {exc}')


## 11. Safety sweep — last-window download (run anytime)

Snapshots completed work into a timestamped safety ZIP, rebuilds the small results pack over it, and exposes both for download. Run this cell at any time — especially before the session ends or after any failure — so completed experiments are never lost.

In [ ]:
import subprocess as _sp
_snap = _sp.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'safety-snapshot', '--run-dir', str(RUN_DIR), '--reason', 'notebook-safety-sweep'],
    cwd=REPO, capture_output=True, text=True, timeout=300)
if _snap.stdout:
    print(_snap.stdout)
RESULTS_ZIP = RUN_ROOT / f'{RUN_ID}-results.zip'
if RESULTS_ZIP.is_file():
    try:
        RESULTS_ZIP.unlink()
    except OSError:
        pass
_pack_args = ['package-results', '--run-dir', str(RUN_DIR), '--out', str(RESULTS_ZIP), '--run-id', RUN_ID]
for _extra in (BUILD_REPORT_DIR, EXPORT_DIR):
    _pack_args += ['--extra', str(_extra)]
for _xp in sorted(RUN_ROOT.glob('xprobe-*')):
    _pack_args += ['--extra', str(_xp)]
run_k8('package results (safety sweep)', *_pack_args)
print('SAFETY COMPLETE — download below, then Save a Version.')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULTS_ZIP)))
    _latest = RUN_DIR / 'safety' / 'safety-latest.json'
    if _latest.is_file():
        _ptr = json.loads(_latest.read_text(encoding='utf-8'))
        _szip = RUN_DIR / 'safety' / Path(_ptr.get('archive', '')).name
        if _szip.is_file():
            display(FileLink(str(_szip)))
except Exception as exc:
    print(f'Use Kaggle Output to download the ZIPs: {exc}')
